# End-to-End Face Mask Classification Pipeline
Notebook ini mencakup:
1. SVM Tanpa Augmentasi (Canny & DWT)
2. SVM Dengan Augmentasi On-the-fly (Canny & DWT)
3. MobileNetV2 Tanpa Augmentasi
4. MobileNetV2 Dengan Augmentasi


In [ ]:
import sys
import os
# Jika script diupload sebagai dataset, hilangkan tanda pagar di bawah ini dan sesuaikan nama foldernya
sys.path.append('/kaggle/input/datasets/emageeeee/pcd-k23')

import cv2
from config import *
from src.preprocess import download_data_from_kaggle, mobilenet_preprocessing, setup_generators
from src.feature_engineering import load_and_extract_features, scale_features, extract_canny, extract_dwt
from src.train import train_svm, train_mobilenet
from src.evaluate import evaluate_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

MODELS_DIR.mkdir(parents=True, exist_ok=True)

kaggle_dir = download_data_from_kaggle()
print("Data terhubung di:", kaggle_dir)

## Skenario 1: SVM Tanpa Augmentasi (Canny & DWT)

In [ ]:
print("Loading & Extracting Train data (Unaugmented)...")
X_train_c, X_train_d, y_train = load_and_extract_features(kaggle_dir / "Train", require_preprocess=True)

print("\nLoading & Extracting Validation data...")
X_val_c, X_val_d, y_val = load_and_extract_features(kaggle_dir / "Validation", require_preprocess=True)

print("\nScaling Features (Canny & DWT)...")
X_train_c_s, X_val_c_s, _, _ = scale_features(X_train_c, X_val_c, None)
X_train_d_s, X_val_d_s, _, _ = scale_features(X_train_d, X_val_d, None)

print("\nMenyimpan Fitur Ekstraksi (Unaugmented)...")
np.savez_compressed(MODELS_DIR / 'features_unaug.npz', X_train_c=X_train_c_s, X_train_d=X_train_d_s, y_train=y_train, X_val_c=X_val_c_s, X_val_d=X_val_d_s, y_val=y_val)

print("\n--- Training & Evaluasi SVM (Fitur Canny - Unaugmented) ---")
svm_model_c = train_svm(X_train_c_s, y_train, MODELS_DIR / 'svm_canny_unaug.pkl')
pred_svm_c = svm_model_c.predict(X_val_c_s)
evaluate_model(y_val, pred_svm_c, CLASSES, "SVM (Canny Unaugmented)")

print("\n--- Training & Evaluasi SVM (Fitur DWT - Unaugmented) ---")
svm_model_d = train_svm(X_train_d_s, y_train, MODELS_DIR / 'svm_dwt_unaug.pkl')
pred_svm_d = svm_model_d.predict(X_val_d_s)
evaluate_model(y_val, pred_svm_d, CLASSES, "SVM (DWT Unaugmented)")

## Skenario 2: SVM Dengan Augmentasi (Canny & DWT)

In [ ]:
print("Menyiapkan Data Augmentasi On-the-fly untuk SVM...")
train_datagen_svm, _ = setup_generators(kaggle_dir)
train_gen_svm = train_datagen_svm.flow_from_directory(
    kaggle_dir / "Train", target_size=IMG_SIZE_SVM, batch_size=BATCH_SIZE, 
    class_mode='categorical', shuffle=True
)

X_train_aug_c, X_train_aug_d, y_train_aug = [], [], []
total_batches = 313 # Sesuai dengan ~10000 gambar / 32 batch
print(f"Mengekstrak fitur Canny & DWT dari {total_batches} batch hasil augmentasi...")

for i in range(total_batches):
    batch_x, batch_y = next(train_gen_svm)
    for j in range(len(batch_x)):
        img_uint8 = (batch_x[j] * 255.0).astype(np.uint8)
        # FIX: Ensure the image is converted strictly to 2D grayscale
        if len(img_uint8.shape) == 3 and img_uint8.shape[-1] == 3:
            import cv2
            img_gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
        else:
            img_gray = img_uint8.squeeze()
            
        X_train_aug_c.append(extract_canny(img_gray))
        X_train_aug_d.append(extract_dwt(img_gray))

        y_train_aug.append(np.argmax(batch_y[j]))

X_train_aug_c = np.array(X_train_aug_c, dtype=np.float32)
X_train_aug_d = np.array(X_train_aug_d, dtype=np.float32)
y_train_aug = np.array(y_train_aug)

print("Scaling Fitur Augmentasi...")
X_train_aug_c_s, _, _, _ = scale_features(X_train_aug_c, None, None)
X_train_aug_d_s, _, _, _ = scale_features(X_train_aug_d, None, None)

print("\nMenyimpan Fitur Ekstraksi (Augmented)...")
np.savez_compressed(MODELS_DIR / 'features_aug.npz', X_train_aug_c=X_train_aug_c_s, X_train_aug_d=X_train_aug_d_s, y_train_aug=y_train_aug)

print("\n--- Training & Evaluasi SVM (Fitur Canny - Augmented) ---")
svm_model_aug_c = train_svm(X_train_aug_c_s, y_train_aug, MODELS_DIR / 'svm_canny_aug.pkl')
pred_svm_aug_c = svm_model_aug_c.predict(X_val_c_s)
evaluate_model(y_val, pred_svm_aug_c, CLASSES, "SVM (Canny Augmented)")

print("\n--- Training & Evaluasi SVM (Fitur DWT - Augmented) ---")
svm_model_aug_d = train_svm(X_train_aug_d_s, y_train_aug, MODELS_DIR / 'svm_dwt_aug.pkl')
pred_svm_aug_d = svm_model_aug_d.predict(X_val_d_s)
evaluate_model(y_val, pred_svm_aug_d, CLASSES, "SVM (DWT Augmented)")

## Setup Generator CNN

In [ ]:
datagen_train_aug = ImageDataGenerator(
    rotation_range=10, width_shift_range=0.2, height_shift_range=0.2,
    zoom_range=0.25, horizontal_flip=True, preprocessing_function=mobilenet_preprocessing
)
datagen_unaug = ImageDataGenerator(preprocessing_function=mobilenet_preprocessing)

val_gen_cnn = datagen_unaug.flow_from_directory(
    kaggle_dir / "Validation", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

## Skenario 3: MobileNetV2 Tanpa Augmentasi

In [ ]:
print("\n--- Training MobileNetV2 (TANPA Augmentasi) ---")
train_gen_unaug_cnn = datagen_unaug.flow_from_directory(
    kaggle_dir / "Train", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical'
)

mobilenet_model_unaug, _ = train_mobilenet(
    train_gen_unaug_cnn, val_gen_cnn, MODELS_DIR / 'mobilenet_unaug.h5', epochs=5
)

print("\nEvaluasi MobileNetV2 (TANPA Augmentasi)...")
pred_cnn_unaug_probs = mobilenet_model_unaug.predict(val_gen_cnn)
pred_cnn_unaug = np.argmax(pred_cnn_unaug_probs, axis=1)
evaluate_model(val_gen_cnn.classes, pred_cnn_unaug, CLASSES, "MobileNetV2 Unaugmented")

## Skenario 4: MobileNetV2 Dengan Augmentasi

In [ ]:
print("\n--- Training MobileNetV2 (DENGAN Augmentasi) ---")
train_gen_aug_cnn = datagen_train_aug.flow_from_directory(
    kaggle_dir / "Train", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical'
)

mobilenet_model_aug, _ = train_mobilenet(
    train_gen_aug_cnn, val_gen_cnn, MODELS_DIR / 'mobilenet_aug.h5', epochs=5
)

print("\nEvaluasi MobileNetV2 (DENGAN Augmentasi)...")
pred_cnn_aug_probs = mobilenet_model_aug.predict(val_gen_cnn)
pred_cnn_aug = np.argmax(pred_cnn_aug_probs, axis=1)
evaluate_model(val_gen_cnn.classes, pred_cnn_aug, CLASSES, "MobileNetV2 Augmented")